# 第5章 条件、循环与批处理

对应正文前置能力：能读懂后续 notebook 中的筛选、循环、列表推导式和小批量处理。

本 notebook 继续使用 `data/small/planetary_orbits_demo.csv`。我们不追求复杂算法，而是把后续科研代码里反复出现的控制流模式先练熟：什么时候用 `if`，什么时候用 `for`，什么时候把一组对象批量处理成结构化结果。

## 学习目标

- 理解 `if`、`elif`、`else` 如何把科学规则写成代码规则。
- 使用 `for` 循环、累加器和列表推导式完成样本筛选与统计。
- 理解“批处理”的基本思想：同一段逻辑稳定地作用在一组对象上。
- 能在循环中加入轻量检查，发现缺失字段、异常取值和空样本。

## 载入并整理数据

CSV 文件里的内容最初都是字符串。进入条件判断和数值比较之前，先把需要计算的字段转换成浮点数。这个动作在后续 FITS 表、光变曲线、光谱样本和机器学习特征表中都会反复出现。

In [ ]:
import csv
from pathlib import Path

data_path = Path("../../data/small/planetary_orbits_demo.csv")
numeric_fields = ["semi_major_axis_au", "orbital_period_years", "mass_earth", "radius_earth"]

with data_path.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    for field in numeric_fields:
        row[field] = float(row[field])

print("样本数:", len(rows))
print("第一条记录:", rows[0])

## 用条件判断表达科学规则

条件判断的本质，是把“如果某个科学条件成立，就采取某种处理方式”写成可重复执行的规则。下面的分类规则很粗糙，但足以说明 `if` / `elif` / `else` 的阅读方式。

In [ ]:
classified_rows = []

for row in rows:
    if row["semi_major_axis_au"] < 2.0 and row["radius_earth"] < 2.0:
        group = "inner_rocky_candidate"
    elif row["mass_earth"] > 10.0:
        group = "giant_planet_candidate"
    else:
        group = "transition_or_edge_case"

    classified_rows.append({
        "planet": row["planet"],
        "group": group,
        "period_years": row["orbital_period_years"],
    })

for item in classified_rows:
    print(item["planet"], "->", item["group"])

## 布尔表达式与样本筛选

后续很多章节都会出现类似写法：从一个样本表中选出满足条件的行，再对这部分样本作图、拟合或评估模型。列表推导式不是魔法，它只是把“建立空列表、循环、判断、追加”压缩成一行。

In [ ]:
inner_rocky = [
    row for row in rows
    if row["semi_major_axis_au"] <= 1.6 and row["radius_earth"] < 2.0
]

large_planets = [
    row for row in rows
    if row["mass_earth"] >= 10.0 or row["radius_earth"] >= 4.0
]

print("内侧岩质候选:", [row["planet"] for row in inner_rocky])
print("大型行星候选:", [row["planet"] for row in large_planets])

## 循环中的累加器

累加器是科研代码中最常见的循环模式之一：一边遍历样本，一边更新当前统计量。这里我们同时计算平均轨道周期，并找出周期最长和最短的行星。

In [ ]:
period_sum = 0.0
shortest = None
longest = None

for row in rows:
    period = row["orbital_period_years"]
    period_sum = period_sum + period

    if shortest is None or period < shortest["orbital_period_years"]:
        shortest = row
    if longest is None or period > longest["orbital_period_years"]:
        longest = row

mean_period = period_sum / len(rows)
print("平均轨道周期:", round(mean_period, 2), "year")
print("最短周期:", shortest["planet"], shortest["orbital_period_years"])
print("最长周期:", longest["planet"], longest["orbital_period_years"])

## 批处理：同一规则作用在多个阈值上

批处理不一定意味着处理成千上万个文件。只要我们把同一段逻辑稳定地应用到一组对象、参数或阈值上，就已经在做批处理。下面用不同半长轴阈值统计样本数量。

In [ ]:
axis_limits = [0.5, 1.0, 2.0, 10.0, 30.0]
summary = []

for limit in axis_limits:
    selected = [row for row in rows if row["semi_major_axis_au"] <= limit]
    summary.append({
        "axis_limit_au": limit,
        "n_selected": len(selected),
        "planets": [row["planet"] for row in selected],
    })

for item in summary:
    print(f"a <= {item['axis_limit_au']:>4.1f} AU: {item['n_selected']} planets {item['planets']}")

## `while` 循环：直到条件满足才停止

`for` 循环适合“把一组对象全部看一遍”。`while` 循环更适合“不断尝试，直到某个条件满足”。科研代码中它常出现在迭代求解、重试任务、轮询状态等场景。这里用一个很小的例子：从内向外扫描，直到找到第一个质量超过地球 10 倍的行星。

In [ ]:
ordered_rows = sorted(rows, key=lambda row: row["semi_major_axis_au"])
index = 0
first_large = None

while first_large is None and index < len(ordered_rows):
    candidate = ordered_rows[index]
    if candidate["mass_earth"] > 10.0:
        first_large = candidate
    index = index + 1

print("第一个大质量行星:", first_large["planet"])
print("检查过的记录数:", index)

## 循环中的轻量检查

先修阶段不需要学习完整测试框架，但要养成一个习惯：循环处理之前先检查字段，循环处理之后检查结果是否为空。很多科研代码的错误不是算法错，而是输入数据没有满足预期。

In [ ]:
required_fields = {"planet", "semi_major_axis_au", "orbital_period_years"}
missing = required_fields - set(rows[0])
if missing:
    raise ValueError(f"缺少必要字段: {sorted(missing)}")

outer_planets = [row for row in rows if row["semi_major_axis_au"] > 2.0]
if not outer_planets:
    raise ValueError("筛选结果为空，请检查阈值是否过严。")

print("外侧行星:", [row["planet"] for row in outer_planets])

## 练习建议

1. 把“大型行星候选”的规则改成 `mass_earth > 50`，观察分类结果如何变化。
2. 用循环计算所有行星的平均半径，并找出半径最大的行星。
3. 增加一个阈值列表，批量统计 `orbital_period_years <= limit` 的样本数。
4. 故意把某个字段名写错，观察轻量检查如何帮助你定位问题。

## 小结

读懂后续 notebook 的关键，不是背语法，而是识别控制流模式：筛选样本、遍历样本、更新统计量、对一组阈值重复同一段逻辑、在关键位置检查输入和输出。只要这些模式熟悉了，后面的 FITS 批处理、HR 图筛选、光变曲线周期扫描和机器学习数据划分都会容易许多。